In [0]:
import os
import pandas as pd
#import geopandas as gpd
import numpy as np
import matplotlib as plt
import seaborn as sns
#from shapely import Point

In [0]:
#put crime data table into a dataframe
mers_data = spark.table('crime_data.bronze.bronze_merseyside_crime').toPandas()

#put boundary data table into a dataframe
#mers_boundary = spark.table('crime_data.bronze.bronze_merseyside_boundary').toPandas()

# Auditing

In [0]:
#Check percentage of nulls in each column
def null_percent_checker(df):
    x=0
    for i in range(0,len(df.columns)):
        print('Percentage of nulls in column '+df.columns[x]+' =',
        len(df[df[df.columns[x]].isnull()==True])/len(df[df.columns[x]])*100,'%')
        x=x+1
null_percent_checker(mers_data)
#The 'Context' column contains only nulls
#The 'Crime ID' and the 'Last outcome category' columns both contain the same percentage of nulls, suggesting a common cause.

In [0]:
#Check whether null 'Crime ID' and 'Last outcome category' values have any values in common in other categories
mers_data[mers_data['crime_id'].isnull() & mers_data['last_outcome_category'].isnull()].nunique()
#When both 'Crime ID' and 'Last outcome category' are null, it is only with one value in 'Reported by', 'Falls within' and 'Crime type'
#Since 'Reported by' and 'Falls within' only have one value in the whole dataset, the issue seems to be with the 'Crime type' value

In [0]:
#Check the data where both 'Crime ID' and 'Last outcome category' are null
mers_data[mers_data['crime_id'].isnull() & mers_data['last_outcome_category'].isnull()]['crime_type']
#The 'Crime type' value where both 'Crime ID' and 'Last outcome category' are null is 'Anti-social behaviour'
#This may be because anti-social behaviour does not count as a crime and as such does not have a crime id or last outcome category

In [0]:
#Check whether all location values are within the PFA boundary for Merseyside
#def show_points_in_pfa(df,boundary):
    #print('Loading...')
    #geometry = [Point(xy) for xy in zip(df['Longitude'], df['Latitude'])]
    #gdf = gpd.GeoDataFrame(df, geometry=geometry)
    #gdf = gdf.set_crs(epsg=4326)
    
    #ax = boundary.plot(figsize=(10,10),color='green',zorder=1)
    #sj = gdf.sjoin(boundary,how='left',predicate='intersects')
    
    #sj.loc[gpd.pd.isna(sj.index_right)].plot(ax=ax,markersize=2,color='red',zorder=2)
    #sj.loc[~gpd.pd.isna(sj.index_right)].plot(ax=ax,markersize=2,color='blue',zorder=2)
    #print('Loaded')

#show_points_in_pfa(mers_data,mers_boundary)

In [0]:
#Check for duplicates
print(mers_data[mers_data.duplicated()].nunique())
#It seems that all duplicated columns have the same crime type, reported_by and falls_within values.
#Given that reported_by and falls_within are the same for all rows, it seems that the common factor is the crime type.
print(mers_data[mers_data.duplicated()]['crime_type'].unique())
#The common crime type is anti-social behaviour
print(mers_data[mers_data.duplicated()].shape)
print(mers_data[mers_data['crime_type']=='Anti-social behaviour'].shape)
#There are 23,348 duplicated rows out of 71,986 total rows for the anti-social behaviour crime type.

# Cleaning

In [0]:
#Remove the context column
mers_data.drop('Context',axis=1,inplace=True)

In [0]:
#Remove duplicate values
mers_data.drop_duplicates(inplace=True)

In [0]:
#Split years and months into two separate columns
def split_ym(df):
    split = df['Month'].str.split('-',expand=True)
    split.columns = ['year','month']
    
    df = df.drop('Month',axis=1)
    df.insert(1,'month',split['month'])
    df.insert(1,'year',split['year'])
    return df

mers_data = split_ym(mers_data)

In [0]:
#Remove records that are located outside the PFA boundary
#def remove_outside_points(df, boundary):
    #geometry = [Point(xy) for xy in zip(df['Longitude'], df['Latitude'])]
    #gdf = gpd.GeoDataFrame(df, geometry=geometry)
    #gdf = gdf.set_crs(epsg=4326)
    #sj = gdf.sjoin(boundary,how='left',predicate='intersects')
    #cleaned = sj.loc[~pd.isna(sj.index_right)].drop(columns=['index_right','id','Name','description','timestamp','begin','end','altitudeMode','tessellate','extrude','visibility','drawOrder','icon','geometry'])
    #return cleaned

#mers_data = remove_outside_points(mers_data,mers_boundary)

In [0]:
#Replace NA values
mers_data.fillna({'crime_id':'ASB not recorded as crime'},inplace=True)
mers_data.fillna({'last_outcome_category':'ASB not recorded as crime'},inplace=True)

In [0]:
mers_data.drop(['crime_id','reported_by','Latitude','Longitude','last_outcome_category','Location'],axis=1,inplace=True)

In [0]:
#This needs to be put into the silver table
mers_clean = mers_data

In [0]:
spark.createDataFrame(mers_clean).write.mode('overwrite').saveAsTable('crime_data.silver.silver_merseyside_crime')